In [7]:
%config Completer.use_jedi = False

In [8]:
import numpy as np
import pandas as pd

In [9]:
import os
print(os.getcwd())
print(os.listdir()) # This will show you all files/folders in your current path

H:\Notebooks\Final_Year_Project
['.ipynb_checkpoints', 'cleaned_hamrobazaar_land.csv', 'cleaned_nepali_houses.csv', 'cleaned_nepali_land.csv', 'hamrobazaar_land_for_sale_kathmandu.csv', 'Nepali_house_dataset.csv', 'nepali_land_data.csv', 'nepal_realestate_cleaning -- version-1.ipynb', 'Untitled.ipynb']


In [20]:
pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)



In [281]:
df1=pd.read_csv('cleaned_hamrobazaar_land.csv')
temp_df1=df1.copy()
df3=pd.read_csv('cleaned_nepali_houses.csv')
temp_df3=df3.copy()
df2=pd.read_csv('cleaned_nepali_land.csv')
temp_df2=df2.copy()

In [11]:
df1.head(2)

,ad_id,title,location_raw,price_npr,land_size_aana,road_size_clean,district
0,HB-449254,commercial land for sale in tokha,"Tokha-3, Kathmandu, Bagmati Pradesh",7000000.0,3.0,Above 20 Ft,Kathmandu
1,HB-DC6CF2,Land for sale near kmc hospital,"Kmc Hospital, Changunarayan-2, Bhaktapur",6500000.0,3.0,NaN,Bhaktapur


In [14]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3868 entries, 0 to 3867
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ad_id            3868 non-null   object 
 1   title            3868 non-null   object 
 2   location_raw     3868 non-null   object 
 3   price_npr        3868 non-null   float64
 4   land_size_aana   3868 non-null   float64
 5   road_size_clean  3061 non-null   object 
 6   district         3868 non-null   object 
dtypes: float64(2), object(5)
memory usage: 211.7+ KB


In [185]:
def robust_price_fixer(row):
    # 'price_npr' is your original input column
    p = row['raw_price']
    s = row['land_size_aana']
    
    if s <= 0 or np.isnan(p) or np.isnan(s): 
        return np.nan
    
    # 1. Handle Shorthand Lakhs (e.g., 35 instead of 3,500,000)
    if p < 10000:
        return p * 100000 * s
    
    # 2. Logic to detect if 'p' is a Rate or a Total Price
    implied_rate = p / s
    
    # If the implied rate is less than 10 Lakhs/aana, 
    # the seller almost certainly meant 'p' as the PRICE PER AANA.
    if implied_rate < 1000000 and p < 15000000:
        return p * s
    
    # 3. Otherwise, it's already the total price
    return p

In [184]:
temp_df1

,ad_id,title,district,neighborhood,location_raw,land_size_aana,road_type,raw_price,price_per_aana,calculated_total_price,is_price_suspect,is_price_outlier,is_rate_outlier,is_large_plot
0,HB-449254,commercial land for sale in tokha,Kathmandu,Tokha,"Tokha-3, Kathmandu, Bagmati Pradesh",3.000000,Above 20 Ft,7.000000e+06,2.333333e+06,7.000000e+06,False,False,False,False
1,HB-DC6CF2,Land for sale near kmc hospital,Bhaktapur,Changunarayan,"Kmc Hospital, Changunarayan-2, Bhaktapur",3.000000,NaN,6.500000e+06,2.166667e+06,6.500000e+06,False,False,False,False
2,HB-72A275,10 Aana Land On Sale At Shivapuri Colony.,Kathmandu,बूढानीलकण्ठ,"बूढानीलकण्ठ, बूढानीलकण्ठ नगरपालिका, काठमाडौं ज...",10.000000,13-20 Ft,5.300000e+06,5.300000e+06,5.300000e+07,False,False,False,False
3,HB-6CE9DF,Land on sale,Lalitpur,Godawari,"Godawari Bridge, Gwarko-Lamatar, Mahalaxmi-8, ...",3.500000,NaN,3.000000e+06,3.000000e+06,1.050000e+07,False,False,False,False
4,HB-5E933A,15 Aana Land On Sale Near Lemon Tree Hotel.,Kathmandu,बूढानीलकण्ठ,"बूढानीलकण्ठ, बूढानीलकण्ठ नगरपालिका, काठमाडौं ज...",15.000000,13-20 Ft,4.500000e+06,4.500000e+06,6.750000e+07,False,False,False,False
5,HB-B4FBC4,Kapan Tenzin Chok Ma 6anna Jagga Sell Ma,Kathmandu,Budhanilkantha,"F84, Tenzing Chok, Budhanilkantha, Budhanilkan...",6.000000,13-20 Ft,6.000000e+06,1.000000e+06,6.000000e+06,False,False,False,False
6,HB-3A8E36,Land For Sale At Chandragiri,Kathmandu,Matatirtha,"Matatirtha, Chandragiri-6, Kathmandu, Bagmati ...",6.000000,Gravel Road,2.500000e+06,2.500000e+06,1.500000e+07,False,False,False,False
7,HB-500628,land for sale bhanimandal lalitpur 10 aana,Lalitpur,दमकल सुन्दरीघाट मार्ग,"DAV Sushil Kedia Vishwa Bharati School, दमकल स...",10.000000,Above 20 Ft,1.050000e+06,1.050000e+06,1.050000e+07,False,False,False,False
8,HB-6E7D12,6 aana perfect land on sale near Metro apartment,Kathmandu,Swastik Marg,"Swastik Marg, Kathmandu-14, Kathmandu, Bagmati...",6.000000,NaN,9.000000e+06,1.500000e+06,9.000000e+06,False,False,False,False
9,HB-D76F6F,land for sale jawalakhel lalitpur 6 aana,Lalitpur,Ekantakuna,"Damodar Marga, Ekantakuna, Lalitpur, Lalitpur ...",6.000000,Above 20 Ft,1.100000e+07,1.833333e+06,1.100000e+07,False,False,False,False


In [186]:
# 1. Create the corrected Total Price column
temp_df1['total_price_npr'] = temp_df1.apply(robust_price_fixer, axis=1)

# 2. Calculate the Rate (This will now be 28 Lakh for Row 1913, not 66k)
temp_df1['price_per_aana'] = temp_df1['total_price_npr'] / temp_df1['land_size_aana']

In [187]:
# 1. Flag Suspect Rows (shorthand placeholders)

# 2. Flag Large Plots
temp_df1['is_large_plot'] = temp_df1['land_size_aana'] > 50

# 3. Flag Rate Outliers (The junk/garbage filter)
# Now Row 1913 will pass (is_rate_outlier = False)

# 4. Flag Total Price Outliers


In [188]:
temp_df1[temp_df1['is_price_suspect']=='True']

,ad_id,title,district,neighborhood,location_raw,land_size_aana,road_type,raw_price,price_per_aana,calculated_total_price,is_price_suspect,is_price_outlier,is_rate_outlier,is_large_plot,total_price_npr


In [189]:
df1[df1['is_price_outlier']]

,ad_id,title,district,neighborhood,location_raw,land_size_aana,road_type,raw_price,price_per_aana,calculated_total_price,is_price_suspect,is_price_outlier,is_rate_outlier,is_large_plot


In [190]:
df1[df1['district']=='Unknown']


,ad_id,title,district,neighborhood,location_raw,land_size_aana,road_type,raw_price,price_per_aana,calculated_total_price,is_price_suspect,is_price_outlier,is_rate_outlier,is_large_plot


In [191]:
import pandas as pd

# Define the mapping for Nepali/English keywords to Districts
district_map = {
    # KATHMANDU
    'काठमाडौं': 'Kathmandu', 'काठमाडौँ': 'Kathmandu', 'Kathmandu': 'Kathmandu',
    'बूढानीलकण्ठ': 'Kathmandu', 'बुढानीलकण्ठ': 'Kathmandu', 'Budhanilkantha': 'Kathmandu',
    'टोखा': 'Kathmandu', 'Tokha': 'Kathmandu',
    'कीर्तिपुर': 'Kathmandu', 'Kirtipur': 'Kathmandu',
    'नागार्जुन': 'Kathmandu', 'Nagarjun': 'Kathmandu',
    'तारकेश्वर': 'Kathmandu', 'Tarakeshwar': 'Kathmandu',
    'गोकर्णेश्वर': 'Kathmandu', 'Gokarneshwar': 'Kathmandu',
    'दक्षिणकाली': 'Kathmandu', 'Dakshinkali': 'Kathmandu',
    'चन्द्रागिरी': 'Kathmandu', 'Chandragiri': 'Kathmandu',
    'शङ्खरापुर': 'Kathmandu', 'Shankharapur': 'Kathmandu',
    'Sitapaila': 'Kathmandu', 'Swayambhu': 'Kathmandu',

    # LALITPUR
    'ललितपुर': 'Lalitpur', 'Lalitpur': 'Lalitpur',
    'महालक्ष्मी': 'Lalitpur', 'Mahalaxmi': 'Lalitpur',
    'गोदावरी': 'Lalitpur', 'Godawari': 'Lalitpur',
    'Imadol': 'Lalitpur', 'Tikathali': 'Lalitpur', 'Thecho': 'Lalitpur',
    'Jharuwarasi': 'Lalitpur', 'Gwarko': 'Lalitpur',

    # BHAKTAPUR
    'भक्तपुर': 'Bhaktapur', 'Bhaktapur': 'Bhaktapur',
    'चाँगुनारायण': 'Bhaktapur', 'Changunarayan': 'Bhaktapur',
    'मध्यपुर': 'Bhaktapur', 'Madhyapur': 'Bhaktapur',
    'सूर्यविनायक': 'Bhaktapur', 'Suryabinayak': 'Bhaktapur',
    'Thimi': 'Bhaktapur', 'Duwakot': 'Bhaktapur', 'Lokanthali': 'Bhaktapur'
}

def resolve_district(row):
    # Only process if currently 'Unknown'
    if row['district'] == 'Unknown':
        # Combine title and location for a wider search area
        search_text = f"{str(row['title'])} {str(row['location_raw'])}"
        
        for key, value in district_map.items():
            if key in search_text:
                return value
                
    return row['district']

# Apply the fix
temp_df1['district'] = temp_df1.apply(resolve_district, axis=1)

# Final check to see how many 'Unknown' remain
print("Updated District Counts:")
print(temp_df1['district'].value_counts())

Updated District Counts:
district
Kathmandu    2592
Lalitpur     1076
Bhaktapur     200
Name: count, dtype: int64


In [192]:
temp_df1[temp_df1['land_size_aana']>50].shape

(157, 15)

In [195]:
# 1. Calculate Price per Aana (Rate)
# Since price_npr is now the total price, we divide by size
temp_df1['price_per_aana'] = temp_df1['total_price_npr'] / temp_df1['land_size_aana']

# 2. Flag Large Plots (> 50 Aana)
# Even if the price is correct, these represent a different market (Commercial/Bulk)
temp_df1['is_large_plot'] = temp_df1['land_size_aana'] > 50

# 3. Flag Rate Outliers (Price per Aana)
# In the KTM Valley context:
# - Under 5 Lakhs/aana: Highly likely to be junk data or extreme rural outskirts.
# - Over 1.5 Crore/aana: Likely commercial luxury or a decimal point error.
temp_df1['is_rate_outlier'] = (temp_df1['price_per_aana'] < 500000) | (temp_df1['price_per_aana'] > 150000000)

# --- Summary Statistics ---
print(f"Total rows: {len(temp_df1)}")
print(f"Large Plots identified: {temp_df1['is_large_plot'].sum()}")
print(f"Suspicious Rates identified: {temp_df1['is_rate_outlier'].sum()}")

# Let's see some of the 'Suspect' rows to confirm they are actually garbage
print("\nPreview of Suspect Rate Rows:")
print(temp_df1[temp_df1['is_rate_outlier'] == True][['title', 'price_npr', 'land_size_aana', 'price_per_aana']].head(10))

Total rows: 3868
Large Plots identified: 157
Suspicious Rates identified: 38

Preview of Suspect Rate Rows:


KeyError: "['price_npr'] not in index"

In [89]:
# This converts the column to a formatted string just for the print output
# Use .map to apply formatting to every individual cell in these columns
# print(df1[['price_npr', 'price_per_aana']].map(lambda x: '{:,.0f}'.format(x)))

In [196]:
temp_df1.sample(3)

,ad_id,title,district,neighborhood,location_raw,land_size_aana,road_type,raw_price,price_per_aana,calculated_total_price,is_price_suspect,is_price_outlier,is_rate_outlier,is_large_plot,total_price_npr
3062,HB-50B3F7,19 Aana Land on sale at Maharajgung kathmandu.,Kathmandu,Maharajgunj,"Maharajgunj, Kathmandu",19.0,NaN,8000000.0,8000000.0,152000000.0,False,False,False,False,152000000.0
2828,HB-9F5A74,Land on sell at Batisputali,Kathmandu,Baneshwar,"battishputali purano baneshwar, Battisputali (...",6.0,Pitched Road,40200000.0,6700000.0,40200000.0,False,False,False,False,40200000.0
2221,HB-57A232,"Land on sale at Gyaneshwor,Kathmandu",Kathmandu,Gyaneswor,"Gyaneswor, Gyaneshwar, Kathmandu",19.0,Pitched Road,8500000.0,8500000.0,161500000.0,False,False,False,False,161500000.0


In [197]:
temp_df1.head(2)

,ad_id,title,district,neighborhood,location_raw,land_size_aana,road_type,raw_price,price_per_aana,calculated_total_price,is_price_suspect,is_price_outlier,is_rate_outlier,is_large_plot,total_price_npr
0,HB-449254,commercial land for sale in tokha,Kathmandu,Tokha,"Tokha-3, Kathmandu, Bagmati Pradesh",3.0,Above 20 Ft,7000000.0,2.333333e+06,7000000.0,False,False,False,False,7000000.0
1,HB-DC6CF2,Land for sale near kmc hospital,Bhaktapur,Changunarayan,"Kmc Hospital, Changunarayan-2, Bhaktapur",3.0,NaN,6500000.0,2.166667e+06,6500000.0,False,False,False,False,6500000.0


In [198]:
import re

def super_clean_neighborhood(name):
    if pd.isna(name): return "Unknown"
    name = str(name).title().strip()

    # 1. Translation & Direct Fixes (Expanding your map)
    # This catches specific misspelled words or Nepali script before any other logic
    direct_fix = {
        'Budanilkantha': 'Budhanilkantha', 'Budhanilakantha': 'Budhanilkantha',
        'भैसेपाटी': 'Bhaisepati', 'Bhaisipati': 'Bhaisepati', 'भैसेपाटी': 'Bhaisepati',
        'Jamsikhel': 'Jhamsikhel', 'झम्सीखेल': 'Jhamsikhel', 'Jamsikhel': 'Jhamsikhel',
        'Kritipur': 'Kirtipur', 'Kritipur Kritipur': 'Kirtipur',
        'Basundhara': 'Vasundhara', 'Basbari': 'Bansbari', 'Bansbari Kathamandu': 'Bansbari',
        'Lazimpat': 'Lajimpat', 'Lajimpat': 'Lajimpat',
        'Kathmandu': 'Unknown', 'Lalitpur': 'Unknown', 'Bhaktapur': 'Unknown',
        'ग्वार्को': 'Gwarko', 'कलंकी': 'Kalanki', 'थानकोट': 'Thankot',
        'Kupondol': 'Kupondole', 'Kopundole': 'Kupondole', 'Kopondole': 'Kupondole',
        'Ekantakuna': 'Ekantakuna', 'एकान्तकुना': 'Ekantakuna'
    }
    
    if name in direct_fix: return direct_fix[name]

    # 2. PRIORITY EXTRACTION (The "Search-and-Destroy" for Messy Strings)
    # We look for these keywords INSIDE the messy strings. 
    # Order matters here: put the most specific names first.
    keywords = [
        'Budhanilkantha', 'Bhaisepati', 'Kirtipur', 'Tokha', 'Baluwatar', 
        'Baneshwor', 'Baneshwar', 'Kalanki', 'Maharajgunj', 'Hattiban', 
        'Dhapakhel', 'Sanepa', 'Jhamsikhel', 'Imadol', 'Tikathali', 
        'Suryabinayak', 'Godawari', 'Chandragiri', 'Nagarjun', 'Dhumbarahi',
        'Sitapaila', 'Golfutar', 'Gokarneshwor', 'Tarakeshwor', 'Chabahil',
        'Kapan', 'Sukedhara', 'Balkhu', 'Samakhushi', 'Baluwatar', 'Naxal',
        'Lazimpat', 'Lajimpat', 'Balaju', 'Koteshwor', 'Thali', 'Mulpani'
    ]

    for key in keywords:
        if key.lower() in name.lower():
            return key # Immediately returns the clean keyword if found inside the mess

    # 3. CLEANING PARENTHESES & NUMBERS
    # Removes "(Tokha)", "(Maharajgunj)", etc.
    name = re.sub(r'\(.*?\)', '', name)
    # Removes Ward Numbers (e.g., "Tokha 3" -> "Tokha")
    name = re.sub(r'\s*\d+\s*$', '', name)
    # Removes Landmark Junk
    name = name.replace('Area', '').replace('City', '').replace('Nepal', '').strip()

    return name

temp_df1['neighborhood_clean'] = temp_df1['neighborhood'].apply(super_clean_neighborhood)

In [199]:
temp_df1.sample(12)

,ad_id,title,district,neighborhood,location_raw,land_size_aana,road_type,raw_price,price_per_aana,calculated_total_price,is_price_suspect,is_price_outlier,is_rate_outlier,is_large_plot,total_price_npr,neighborhood_clean
1999,HB-4AE9A2,7 aana land for sale surya binayak bhaktapur,Bhaktapur,Suryabinayak,"Suryabinayak, Bhaktapur",7.00,NaN,25200000.0,3.600000e+06,25200000.0,False,False,False,False,25200000.0,Suryabinayak
2525,HB-2EF29E,Land on sale Kirtipur machhegaun,Kathmandu,अधिकारी टोल,"F104, अधिकारी टोल, Chandragiri-09, Boshigaun, ...",4.50,Gravel Road,2050000.0,2.050000e+06,9225000.0,False,False,False,False,9225000.0,अधिकारी टोल
259,HB-83F707,Lamatar's Land on Sale,Lalitpur,Lamatar,"Lamatar, Mahalaxmi-8, Lalitpur",5.00,Gravel Road,3050000.0,3.050000e+06,15250000.0,False,False,False,False,15250000.0,Lamatar
3650,HB-360572,"Residential Land On Sale Chapali, Budhanilkantha",Kathmandu,Khadkagaon,"Khadkagaon, Budhanilkantha, Budhanilkantha Mun...",5.50,Pitched Road,350000.0,3.500000e+05,1925000.0,False,False,True,False,1925000.0,Khadkagaon
1943,HB-C00971,Residential 11 Aana Land On Sale At Baluwatar ...,Kathmandu,Baluwatar,"Baluwatar Area, Kathmandu",11.00,9-12 Ft,77000000.0,7.000000e+06,77000000.0,False,False,False,False,77000000.0,Baluwatar
1815,HB-92A671,Residential Land on Sale in Tanglaphat,Kathmandu,Kirtipur,"kritipur, Tyangala Phant (Kirtipur), Kathmandu",4.00,NaN,5000000.0,1.250000e+06,5000000.0,False,False,False,False,5000000.0,Kirtipur
3567,HB-4ED0C8,Land for sale in thali Kathmandu,Kathmandu,Thali,"Thali (Kageshwor), Kathmandu",5.00,Above 20 Ft,2500000.0,2.500000e+06,12500000.0,False,False,False,False,12500000.0,Thali
2361,HB-B5163D,4/4 Aana land on sale at sanobharyang Kathmandu,Kathmandu,बुद्ध चोक,"बुद्ध चोक, सान्तिनगर, Nagarjun-02, नागार्जुन न...",8.00,13-20 Ft,5500000.0,5.500000e+06,44000000.0,False,False,False,False,44000000.0,बुद्ध चोक
1423,HB-E5C330,🌟 Prime Land for Sale in Raniban,Kathmandu,Rani Ban,"Rani Ban, Kathmandu-3, Kathmandu",10.00,Above 20 Ft,5000000.0,5.000000e+06,50000000.0,False,False,False,False,50000000.0,Rani Ban
1345,HB-64A18F,Land for Sale at Kirtipur 6,Kathmandu,Kirtipur,"Kirtipur 6, Kathmandu, Kirtipur-6, Kathmandu",5.00,Pitched Road,2800000.0,2.800000e+06,14000000.0,False,False,False,False,14000000.0,Kirtipur


In [170]:
temp_df1['neighborhood'].value_counts().shape

(1194,)

In [200]:
temp_df1['neighborhood_clean'].value_counts()

neighborhood_clean
Budhanilkantha                                                                                                                       516
Nagarjun                                                                                                                             112
Bhaisepati                                                                                                                           107
Kirtipur                                                                                                                              95
Tokha                                                                                                                                 91
Unknown                                                                                                                               86
Chandragiri                                                                                                                           63
Baluwatar             

In [201]:
temp_df1.drop(columns='neighborhood',inplace=True)

In [202]:
temp_df1.sample(2)

,ad_id,title,district,location_raw,land_size_aana,road_type,raw_price,price_per_aana,calculated_total_price,is_price_suspect,is_price_outlier,is_rate_outlier,is_large_plot,total_price_npr,neighborhood_clean
2607,HB-7BC41A,🌟 Prime Land for Sale in Gokarna! 🌟,Kathmandu,"Gokarna (Gokarneshwar), Kathmandu",3.375,NaN,2200000.0,2200000.0,7425000.0,False,False,False,False,7425000.0,Gokarna
3455,HB-49501A,For sell land at basbari,Kathmandu,"basbari, Bansbari, Kathmandu",10.000,Pitched Road,6200000.0,6200000.0,62000000.0,False,False,False,False,62000000.0,Bansbari


In [203]:
# 1. Rename columns accurately
temp_df1 = temp_df1.rename(columns={
    'road_size_clean': 'road_type', 
    'neighborhood_clean': 'neighborhood',
    'total_price_npr': 'calculated_total_price',
    'price_npr': 'raw_price'
})

# 2. Define the final logical order
# Grouping by Location -> Specs -> Price -> Flags
final_order = [
    'ad_id', 
    'title', 
    'district', 
    'neighborhood', 
    'location_raw', 
    'land_size_aana', 
    'road_type',      
    'raw_price', 
    'price_per_aana', 
    'calculated_total_price',
    'is_price_suspect', 
    'is_price_outlier', 
    'is_rate_outlier', 
    'is_large_plot'
]

# Apply the order
temp_df1 = temp_df1[final_order]

In [156]:
# The "Clean Master" Filter
# We keep rows where price is NOT an outlier AND rate is NOT an outlier AND price is NOT suspect
df1 = df1[
    (df1['is_price_outlier'] == False) & 
    (df1['is_rate_outlier'] == False) & 
    (df1['is_price_suspect'] == False)
].copy()




In [207]:
temp_df1.sample(12)

,ad_id,title,district,neighborhood,location_raw,land_size_aana,road_type,raw_price,price_per_aana,calculated_total_price,calculated_total_price,is_price_suspect,is_price_outlier,is_rate_outlier,is_large_plot
1925,HB-EEDC50,"Land on Sale at Kapan Chunikhel, GJBLOS 3524",Kathmandu,Kopan,"kopan, Kapan, Kathmandu",5.25,Gravel Road,3900000.0,3900000.0,20475000.0,20475000.0,False,False,False,False
914,HB-CD81A6,5aana land for sale Suryadarsan Tokha,Kathmandu,Tokha,"Tokha-3, Kathmandu, Bagmati Pradesh",5.00,13-20 Ft,5200000.0,1040000.0,5200000.0,5200000.0,False,False,False,False
2616,HB-2B7753,land on sale Manbhaban lalitapur 13 aana,Lalitpur,Kumaripati,"Man bhawan Marga, Kumaripati, Itapukhu, Lalitp...",13.00,13-20 Ft,6500000.0,6500000.0,84500000.0,84500000.0,False,False,False,False
1560,HB-8BA9CA,3.5 ANA LAND FOR SALE IN THECHO LALITPUR,Lalitpur,Thecho,"Thecho, Godawari-12, Thecho, Godawari, ललितपुर...",3.50,13-20 Ft,2450000.0,2450000.0,8575000.0,8575000.0,False,False,False,False
3097,HB-0B8C59,27 Aana land on sale at basundhara ring road,Kathmandu,Usha Marble,"Usha marble, Kathmandu Ringroad, Tokha-05, टोख...",27.00,Above 20 Ft,12000000.0,12000000.0,324000000.0,324000000.0,False,False,False,False
300,HB-18D7F8,8.50 aana land on sale Bansbari Kathmandu,Kathmandu,Pipal Marg,"Pipal Marg, Kathmandu-3, Kathmandu, Bagmati Pr...",8.00,Above 20 Ft,7000000.0,7000000.0,56000000.0,56000000.0,False,False,False,False
626,HB-02CDDF,sitapaila taufical ma 56 aana jagga bikrima,Kathmandu,Nagarjun,"27DR017, Nagarjun, Nagarjun Municipality, Kath...",56.00,9-12 Ft,2000000.0,2000000.0,112000000.0,112000000.0,False,False,False,True
3138,HB-6EA9D4,"5 AANA 2 PAISA LAND FOR SALE AT LAMATAR, LALITPUR",Lalitpur,Lamatar,"Lamatar (Mahalaxmi), Lalitpur",5.20,NaN,3000000.0,3000000.0,15600000.0,15600000.0,False,False,False,False
1829,HB-D55300,Land For Sale At Budhanilkantha Bhangal,Kathmandu,Golphutar,"Golphutar, Budhanilkantha, Budhanilkantha Muni...",5.00,13-20 Ft,4300000.0,4300000.0,21500000.0,21500000.0,False,False,False,False
2895,HB-C0C7FA,"4 ana land for sale in Harisiddhi , Lalitpur",Lalitpur,Harisiddhi,"Harisiddhi, Lalitpur",4.00,NaN,2400000.0,2400000.0,9600000.0,9600000.0,False,False,False,False


In [213]:
temp_df1.sample(12)

,ad_id,title,district,neighborhood,location_raw,land_size_aana,road_type,raw_price,price_per_aana,calculated_total_price,calculated_total_price,is_price_suspect,is_price_outlier,is_rate_outlier,is_large_plot
2683,HB-752D22,"land for sale in Swayambhu, Adeshwar plotting!",Kathmandu,Swayambhu,"swayambhu, Swoyambhu, Kathmandu",5.000000,NaN,3800000.0,3.800000e+06,1.900000e+07,1.900000e+07,False,False,False,False
3670,HB-59D965,"Land For Sale at Champi, Lalitput",Lalitpur,Champi,"Champi (Karyabinayak), Lalitpur",62.000000,Gravel Road,74400000.0,1.200000e+06,7.440000e+07,7.440000e+07,False,False,False,True
367,HB-35E876,"Land On Sale At Bohoratar, Balaju",Kathmandu,Bohoratar Road,"Bohoratar Road, Kathmandu-16, Kathmandu",8.500000,Gravel Road,4500000.0,4.500000e+06,3.825000e+07,3.825000e+07,False,False,False,False
1755,HB-4FE3AB,land on sale bhaisepati chunikhel lalitpur 3.5,Lalitpur,Bungamati,"Bungamati, Lalitpur, Lalitpur Metropolitan Cit...",3.000000,Above 20 Ft,3500000.0,1.166667e+06,3.500000e+06,3.500000e+06,False,False,False,False
3397,HB-4ED62E,"Land 4 sale chapagaun,Nakhu corridor road",Lalitpur,Chapagaun,"Chapagaun (Bajrabarahi), Lalitpur",18.115413,NaN,1500000.0,1.500000e+06,2.717312e+07,2.717312e+07,False,False,False,False
1932,HB-C9084C,"5.50 aana land for sale, Khadka Bhadrakali kor...",Kathmandu,खड्कागाउँ,"खड्कागाउँ, Budhanilkantha-04, बूढानीलकण्ठ, बूढ...",5.000000,13-20 Ft,3200000.0,3.200000e+06,1.600000e+07,1.600000e+07,False,False,False,False
2377,HB-BA8FCF,Land for sale Sanaypa 20 anna,Lalitpur,Jhamsikhel,"झम्सीखेल, धोबीघाट, Lalitpur-03, ललितपुर, ललितप...",20.000000,13-20 Ft,7500000.0,7.500000e+06,1.500000e+08,1.500000e+08,False,False,False,False
2245,HB-DF0AEB,Kumaripati Ma 11 Aana Jagga Bikrima,Lalitpur,Kumaripati,"Kumaripati First lane Marga, Kumaripati, Itapu...",11.000000,13-20 Ft,13500000.0,1.227273e+06,1.350000e+07,1.350000e+07,False,False,False,False
2534,HB-FCB45F,4 anna land on sell at dholahiti lalitpur,Lalitpur,Dholahiti,"Dholahiti, Lalitpur",4.000000,Gravel Road,4000000.0,1.000000e+06,4.000000e+06,4.000000e+06,False,False,False,False
2260,HB-3791BB,land for sale Sanaypa 17 anna,Lalitpur,दुम्बेधारा सत्संग मार्ग,"दुम्बेधारा सत्संग मार्ग, अरुन थापा मुर्ति चोक,...",17.000000,13-20 Ft,8000000.0,8.000000e+06,1.360000e+08,1.360000e+08,False,False,False,False


In [224]:
df1.head(2)

,ad_id,title,district,neighborhood,location_raw,land_size_aana,road_type,price_per_aana,calculated_total_price,raw_price,is_price_suspect,is_price_outlier,is_rate_outlier,is_large_plot
0,HB-449254,commercial land for sale in tokha,Kathmandu,Tokha,"Tokha-3, Kathmandu, Bagmati Pradesh",3.0,Above 20 Ft,2.333333e+06,7000000.0,7000000.0,False,False,False,False
1,HB-DC6CF2,Land for sale near kmc hospital,Bhaktapur,Changunarayan,"Kmc Hospital, Changunarayan-2, Bhaktapur",3.0,NaN,2.166667e+06,6500000.0,6500000.0,False,False,False,False


In [221]:
temp_df1 = temp_df1.loc[:, ~temp_df1.columns.duplicated()].copy()

# 2. Rename for the final schema
temp_df1 = temp_df1.rename(columns={
    'raw_price': 'input_price',
    'calculated_total_price': 'total_valuation',
    'price_per_aana': 'rate_per_aana',
    'road_typer': 'road_type' 
})

In [226]:
final_order = [
    'ad_id', 
    'title', 
    'location_raw', 
    'district', 
    'neighborhood', 
    'road_type', 
    'raw_price',
    'land_size_aana',
    'price_per_aana', 
    'calculated_total_price',
    'is_price_suspect', 
    'is_price_outlier', 
    'is_rate_outlier', 
    'is_large_plot'
]

# Apply the order to the temp dataframe
df1 = df1[final_order]

In [227]:
df1.head(2)

,ad_id,title,location_raw,district,neighborhood,road_type,raw_price,land_size_aana,price_per_aana,calculated_total_price,is_price_suspect,is_price_outlier,is_rate_outlier,is_large_plot
0,HB-449254,commercial land for sale in tokha,"Tokha-3, Kathmandu, Bagmati Pradesh",Kathmandu,Tokha,Above 20 Ft,7000000.0,3.0,2.333333e+06,7000000.0,False,False,False,False
1,HB-DC6CF2,Land for sale near kmc hospital,"Kmc Hospital, Changunarayan-2, Bhaktapur",Bhaktapur,Changunarayan,NaN,6500000.0,3.0,2.166667e+06,6500000.0,False,False,False,False


In [219]:
df1 = temp_df1.copy()

print("Success: temp_df1 processed and copied to df1.")
print(f"Final Column List: {df1.columns.tolist()}")

Success: temp_df1 processed and copied to df1.
Final Column List: ['ad_id', 'title', 'district', 'neighborhood', 'location_raw', 'land_size_aana', 'road_type', 'rate_per_aana', 'total_valuation', 'input_price', 'is_price_suspect', 'is_price_outlier', 'is_rate_outlier', 'is_large_plot']


In [231]:
# Check a few large plots to see if valuation = rate * size
print(df1[df1['is_large_plot'] == True][['neighborhood', 'land_size_aana', 'price_per_aana', 'calculated_total_price']].head())

               neighborhood  land_size_aana  price_per_aana  \
31                 Hattiban            55.0       4000000.0   
58             Shankharapur            86.0       1500000.0   
147  Aama Ko Achar Homemade            56.0       2200000.0   
156                  Sanepa           224.0       8000000.0   
207          Budhanilkantha            51.0       3500000.0   

     calculated_total_price  
31             2.200000e+08  
58             1.290000e+08  
147            1.232000e+08  
156            1.792000e+09  
207            1.785000e+08  


In [232]:
# Save the final cleaned version
df1.to_csv('cleaned_hamrobazaar_land_version_2.csv', index=False)

print("File saved successfully as 'cleaned_hamrobazaar_land_version_2.csv'")

File saved successfully as 'cleaned_hamrobazaar_land_version_2.csv'


In [257]:
df1.columns

Index(['ad_id', 'title', 'location_raw', 'district', 'neighborhood',
       'road_type', 'raw_price', 'land_size_aana', 'price_per_aana',
       'calculated_total_price', 'is_price_suspect', 'is_price_outlier',
       'is_rate_outlier', 'is_large_plot'],
      dtype='object')

SECOND FILE CLEANING

In [272]:
df2.sample(12)

,location_raw,facing,price_amount_npr,price_unit,road_width_feet,road_surface,land_size_aana,district
1562,"Budhanilkantha, Kathmandu",South-West,6100000.0,per_aana,14.0,Black Topped,NaN,Kathmandu
335,"Chobhar, Kathmandu",North-East,2800000.0,per_aana,22.0,Black Topped,11.2,Kathmandu
1344,"Imadol, Lalitpur",South-West,4500000.0,per_aana,13.0,Paved,4.1,Lalitpur
1810,"Radhe Radhe, Bhaktapur",West,5500000.0,per_aana,20.0,Gravelled,NaN,Bhaktapur
882,"Bansbari, Kathmandu",West,7500000.0,per_aana,20.0,Black Topped,NaN,Kathmandu
838,"Nagdaha, Lalitpur",South-West,3200000.0,per_aana,20.0,Black Topped,NaN,Lalitpur
1071,"Tyanglafat, Kathmandu",South-East,4500000.0,per_aana,13.0,Dhalan,8.2,Kathmandu
417,"Harisiddhi, Lalitpur",North-West,3900000.0,per_aana,13.0,Paved,NaN,Lalitpur
322,"Ganesh Chowk, Kathmandu",South,3700000.0,total,13.0,Black Topped,NaN,Kathmandu
776,"Dhobighat, Lalitpur",West,8000000.0,per_aana,16.0,Black Topped,NaN,Lalitpur


In [273]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1942 entries, 0 to 2042
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   location_raw      1942 non-null   object 
 1   facing            1905 non-null   object 
 2   price_amount_npr  1942 non-null   float64
 3   price_unit        1942 non-null   object 
 4   road_width_feet   1859 non-null   float64
 5   road_surface      1873 non-null   object 
 6   land_size_aana    728 non-null    float64
 7   district          1942 non-null   object 
dtypes: float64(3), object(5)
memory usage: 136.5+ KB


In [236]:
df2['price_unit'].value_counts()

price_unit
per_aana      1797
total           95
per_dhur        23
per_sqm         16
per_ropani       6
per_kattha       6
per_year         1
Name: count, dtype: int64

In [237]:
df2 =df2[df2['price_unit'] != 'per_year'].copy()

In [251]:
df2 = df2[df2['price_amount_npr'] < 1200000000].copy()

In [259]:
df2['price_unit'].value_counts()

price_unit
per_aana      1797
total           94
per_dhur        23
per_sqm         16
per_ropani       6
per_kattha       6
Name: count, dtype: int64

In [282]:
df2 = df2.rename(columns={
    'price_amount_npr': 'raw_price',
    'road_surface': 'road_type'
})

In [283]:
temp_df2 = temp_df2.rename(columns={
    'price_amount_npr': 'raw_price',
    'road_surface': 'road_type'
})

In [284]:
temp_df2.sample(2)

,location_raw,facing,raw_price,price_unit,road_width_feet,road_type,land_size_aana,district
1858,"Kalanki, Kathmandu",North,4500000.0,per_aana,13.0,Black Topped,0.4,Kathmandu
64,"Kamerotar, Bhaktapur",East,5500000.0,per_aana,20.0,Soil Stabilized,NaN,Bhaktapur


In [285]:
temp_df2['price_unit'].value_counts()

price_unit
per_aana      1797
total           95
per_dhur        23
per_sqm         16
per_ropani       6
per_kattha       6
per_year         1
Name: count, dtype: int64

In [293]:
temp_df2.sample(12)

,location_raw,facing,raw_price,price_unit,road_width_feet,road_type,land_size_aana,district
63,"Sirutar, Bhaktapur",South-East,3500000.0,per_aana,13.0,Paved,NaN,Bhaktapur
900,"Naya Bazar, Kathmandu",South,7500000.0,per_aana,13.0,Black Topped,7.2,Kathmandu
53,"Sirutar, Bhaktapur",East,3700000.0,per_aana,13.0,Paved,4.2,Bhaktapur
480,"Mulpani, Kathmandu",West,3700000.0,per_aana,13.0,Dhalan,NaN,Kathmandu
1589,"Taukhel, Lalitpur",South,2700000.0,per_aana,16.0,Dhalan,3.4,Lalitpur
1888,"Bhangal , Kathmandu",East,5200000.0,per_aana,20.0,Black Topped,5.5,Kathmandu
1971,"Badeli, Kathmandu",North-East,4500000.0,per_aana,NaN,NaN,NaN,Kathmandu
1664,"Mandikhatar, Kathmandu",East,3200000.0,per_aana,13.0,Dhalan,NaN,Kathmandu
89,"Sitapaila, Kathmandu",West,2800000.0,per_aana,13.0,Soil Stabilized,NaN,Kathmandu
669,"Damaitar, Lalitpur",North,NaN,NaN,13.0,Soil Stabilized,NaN,Lalitpur


In [305]:
def process_financials(row):
    p = row['raw_price']
    unit = row['price_unit']
    size = row['land_size_aana']
    
    # Defaults
    rate = np.nan
    total = np.nan
    
    if pd.isna(p):
        return pd.Series([rate, total])

    # Logic for Price Per Aana (The "Rate")
    if unit == 'per_aana':
        rate = p
    elif unit == 'total' and size > 0:
        rate = p / size
    elif unit == 'per_dhur':
        rate = p / 0.5325  # 1 Dhur ≈ 0.5325  Aana
    elif unit == 'per_ropani':
        rate = p / 16
    elif unit == 'per_kattha':
        rate = p / 20
    elif unit == 'per_sqm':
        rate = p * 31.8
        
    # Logic for Calculated Total Price
    if unit == 'total':
        total = p
    elif not pd.isna(rate) and size > 0:
        total = rate * size
        
    return pd.Series([rate, total])

# Apply the math
df2[['price_per_aana', 'calculated_total_price']] = df2.apply(process_financials, axis=1)

In [306]:
df2[df2['price_unit']=='per_ropani']

,location_raw,facing,raw_price,price_unit,road_width_feet,road_type,land_size_aana,district,price_per_aana,calculated_total_price
275,"Sunkhani, Nuwakot",North,1400000.0,per_ropani,12.0,Soil Stabilized,NaN,Unknown,87500.0,NaN
644,"Dachhila, Dhading",East,4100000.0,per_ropani,22.0,Black Topped,NaN,Unknown,256250.0,NaN
823,"Dhulikhel, Kavrepalanchok",South-West,3000000.0,per_ropani,14.0,Soil Stabilized,NaN,Unknown,187500.0,NaN
1827,"Lama Gaun, Nuwakot",NaN,1200000.0,per_ropani,13.0,Gravelled,105.1,Unknown,75000.0,7882500.0
1828,"Kharibhanjyang, Nuwakot",NaN,1000000.0,per_ropani,13.0,Gravelled,108.3,Unknown,62500.0,6768750.0
2029,"Pokhara, Kaski",East,1500000.0,per_ropani,12.0,Black Topped,NaN,Unknown,93750.0,NaN


In [307]:
temp_df2['is_price_suspect'] = temp_df2['raw_price'].isin([0, 1, 1234, 9999, 1111])
temp_df2['is_large_plot'] = temp_df2['land_size_aana'] > 50
temp_df2['is_rate_outlier'] = (temp_df2['price_per_aana'] < 500000) | (temp_df2['price_per_aana'] > 150000000)
temp_df2['is_price_outlier'] = temp_df2['calculated_total_price'] > 500000000 # > 50 Crore

In [315]:
temp_df2.sample(2)


,location_raw,facing,raw_price,price_unit,road_width_feet,road_type,land_size_aana,district,price_per_aana,calculated_total_price,is_price_suspect,is_large_plot,is_rate_outlier,is_price_outlier
1008,"Sanagaun, Lalitpur",South,3200000.0,per_aana,20.0,Gravelled,NaN,Lalitpur,3200000.0,NaN,False,False,False,False
1487,"Raniban, Kathmandu",West,5300000.0,per_aana,22.0,Dhalan,NaN,Kathmandu,5300000.0,NaN,False,False,False,False


In [316]:
df2['is_price_suspect'] = df2['raw_price'].isin([0, 1, 1234, 9999, 1111])
df2['is_large_plot'] = df2['land_size_aana'] > 50
df2['is_rate_outlier'] = (df2['price_per_aana'] < 500000) | (df2['price_per_aana'] > 150000000)
df2['is_price_outlier'] = df2['calculated_total_price'] > 500000000 # > 50 Crore

In [317]:
df2.sample(2)


,location_raw,facing,raw_price,price_unit,road_width_feet,road_type,land_size_aana,district,price_per_aana,calculated_total_price,is_price_suspect,is_large_plot,is_rate_outlier,is_price_outlier
69,"Lamatar, Lalitpur",South,2800000.0,per_aana,13.0,Soil Stabilized,NaN,Lalitpur,2800000.0,NaN,False,False,False,False
1635,"Bhaisepati, Lalitpur",South,4500000.0,per_aana,13.0,Soil Stabilized,NaN,Lalitpur,4500000.0,NaN,False,False,False,False


In [322]:
def clean_neighborhood(row):
    loc = str(row['location_raw'])
    dist = str(row['district'])
    
    # If the location is too long or contains business keywords, 
    # we try to see if a real neighborhood is hidden inside
    noise_keywords = ['homemade', 'achar', 'shop', 'office', 'limited', 'hospital']
    
    # 2. Extract primary neighborhood (first part before comma)
    if ',' in loc:
        primary = loc.split(',')[0].strip()
    else:
        primary = loc.strip()

    # 3. Validation: If the extracted name is just a number or a noise word
    if any(word in primary.lower() for word in noise_keywords) or primary == "":
        return f"Unknown, {dist}"
    
    return primary

# Apply the improved logic
df2['neighborhood'] = df2.apply(clean_neighborhood, axis=1)

# Standardize: Title Case (e.g., "gothatar" -> "Gothatar")
df2['neighborhood'] = df2['neighborhood'].str.title()

In [321]:
df2.sample(2)

,location_raw,facing,raw_price,price_unit,road_width_feet,road_type,land_size_aana,district,price_per_aana,calculated_total_price,is_price_suspect,is_large_plot,is_rate_outlier,is_price_outlier,neighborhood
625,"Hattiban, Lalitpur",West,4800000.0,per_aana,26.0,Dhalan,NaN,Lalitpur,4800000.0,NaN,False,False,False,False,Hattiban
409,"Pasikot, Kathmandu",East,4200000.0,per_aana,13.0,Black Topped,NaN,Kathmandu,4200000.0,NaN,False,False,False,False,Pasikot


In [323]:
# 1. Create the Wide Road Flag
# We use .fillna(False) because if road width is unknown, we can't assume it's wide.
temp_df2['is_wide_road'] = (temp_df2['road_width_feet'] > 40).fillna(False)

# 2. Cleanup: Ensure road_width_feet stays as a float for the model
temp_df2['road_width_feet'] = pd.to_numeric(temp_df2['road_width_feet'], errors='coerce')

# Optional: You can also flag 'very narrow' roads (e.g., < 10 feet) 
# as they usually can't get building permits in KTM.
temp_df2['is_narrow_road'] = temp_df2['road_width_feet'] < 10

In [328]:
temp_df2['facing'] = temp_df2['facing'].str.strip().str.title()

In [329]:
df2['facing'] = df2['facing'].str.strip().str.title()

In [330]:
temp_df2['facing'] = temp_df2['facing'].fillna('Unknown')

In [331]:
df2['facing'] = df2['facing'].fillna('Unknown')

In [333]:
df2.sample(2)

,location_raw,facing,raw_price,price_unit,road_width_feet,road_type,land_size_aana,district,price_per_aana,calculated_total_price,is_price_suspect,is_large_plot,is_rate_outlier,is_price_outlier,neighborhood
1103,"Lokanthali, Bhaktapur",West,5500000.0,per_aana,20.0,Black Topped,NaN,Bhaktapur,5500000.0,NaN,False,False,False,False,Lokanthali
1400,"Narayanthan , Kathmandu",South,4000000.0,per_aana,13.0,Black Topped,NaN,Kathmandu,4000000.0,NaN,False,False,False,False,Narayanthan


In [334]:
print(df2['facing'].value_counts())

facing
South         483
East          472
West          304
North         225
South-East    174
North-East    159
South-West    110
North-West     74
Unknown        42
Name: count, dtype: int64


In [335]:
df2['road_type'] = df2['road_type'].str.strip().str.title().fillna('Unknown')

In [336]:
print(df2['road_type'].value_counts())

road_type
Black Topped       614
Gravelled          389
Soil Stabilized    333
Dhalan             319
Paved              309
Unknown             75
Alley                4
Name: count, dtype: int64


In [337]:
print(df2[['price_per_aana', 'land_size_aana', 'neighborhood', 'district']].isnull().sum())

price_per_aana     152
land_size_aana    1282
neighborhood         0
district             0
dtype: int64


In [338]:
df2.columns

Index(['location_raw', 'facing', 'raw_price', 'price_unit', 'road_width_feet',
       'road_type', 'land_size_aana', 'district', 'price_per_aana',
       'calculated_total_price', 'is_price_suspect', 'is_large_plot',
       'is_rate_outlier', 'is_price_outlier', 'neighborhood'],
      dtype='object')

In [339]:
df2['district'] = df2['district'].astype(str).str.strip().str.title()

In [343]:
df2['district'].isnull().sum()

np.int64(0)

In [344]:
def rescue_district(row):
    current_dist = row['district']
    loc_raw = str(row['location_raw'])
    
    # If district is invalid, try to grab it from location_raw
    if current_dist in ['Nan', 'Unknown', '', 'None']:
        if ',' in loc_raw:
            return loc_raw.split(',')[-1].strip().title()
    return current_dist

In [345]:
df2['district'] = df2.apply(rescue_district, axis=1)

In [348]:
df2['district'].value_counts()

district
Kathmandu         897
Lalitpur          726
Bhaktapur         311
Nuwakot            25
Dhanusa            18
Kavrepalanchok     17
Chitwan            10
Dhading             7
Rupandehi           4
Morang              4
Jhapa               4
Sunsari             3
Kaski               3
Nawalpur            3
Ilam                2
Tanahun             2
Kailali             2
Makwanpur           2
Rautahat            1
Udayapur            1
Palpa               1
Name: count, dtype: int64

In [349]:
# Strip any accidental hidden spaces and force Title Case
df2['district'] = df2['district'].astype(str).str.strip().str.title()

In [350]:
print(df2['road_type'].value_counts())

road_type
Black Topped       614
Gravelled          389
Soil Stabilized    333
Dhalan             319
Paved              309
Unknown             75
Alley                4
Name: count, dtype: int64


In [351]:
road_map = {
    'Black Topped': 'Pitched',
    'Pitched': 'Pitched',
    'Asphalt': 'Pitched',
    'Dhalan': 'Paved',
    'Paved': 'Paved',
    'Concrete': 'Paved',
    'Gravelled': 'Gravel',
    'Soil Stabilized': 'Gravel',
    'Dirt': 'Soil',
    'Soil': 'Soil'
}

df2['road_type'] = df2['road_type'].str.strip().str.title().replace(road_map)

In [352]:
print(df2['road_type'].value_counts())

road_type
Gravel     722
Paved      628
Pitched    614
Unknown     75
Alley        4
Name: count, dtype: int64


In [353]:
# 1. Convert to numeric, forcing errors to NaN (removes 'ft', 'feet', etc.)
temp_df2['road_width_feet'] = pd.to_numeric(temp_df2['road_width_feet'].astype(str).str.extract('(\d+)')[0], errors='coerce')

# 2. Handle zeros
# A road width of 0 is physically impossible for a listed property. 
# We should treat 0 as NaN (Unknown).
temp_df2.loc[df2['road_width_feet'] == 0, 'road_width_feet'] = np.nan

# 3. Apply the 'Wide Road' flag logic we discussed earlier
temp_df2['is_wide_road'] = temp_df2['road_width_feet'] > 40

In [354]:
temp_df2.sample(3)

,location_raw,facing,raw_price,price_unit,road_width_feet,road_type,land_size_aana,district,price_per_aana,calculated_total_price,is_price_suspect,is_large_plot,is_rate_outlier,is_price_outlier,is_wide_road,is_narrow_road
1239,"Shambhu Marga, Kathmandu",East,3800000.0,per_aana,13.0,Dhalan,NaN,Kathmandu,3800000.0,NaN,False,False,False,False,False,False
1357,"Imadol, Lalitpur",North,5500000.0,per_aana,26.0,Paved,NaN,Lalitpur,5500000.0,NaN,False,False,False,False,False,False
1708,"Thimi, Bhaktapur",North,5500000.0,per_aana,26.0,Black Topped,4.3,Bhaktapur,5500000.0,23650000.0,False,False,False,False,False,False


In [366]:
df2.sample(15)

,location_raw,facing,raw_price,price_unit,road_width_feet,road_type,land_size_aana,district,price_per_aana,calculated_total_price,is_price_suspect,is_large_plot,is_rate_outlier,is_price_outlier,neighborhood
261,"Bansbari, Kathmandu",South,6500000.0,per_aana,16.0,Pitched,NaN,Kathmandu,6500000.0,NaN,False,False,False,False,Bansbari
1553,"Harisiddhi, Lalitpur",West,2800000.0,per_aana,13.0,Gravel,3.3,Lalitpur,2800000.0,9240000.0,False,False,False,False,Harisiddhi
1585,"Budhanilkantha, Kathmandu",South-West,4500000.0,per_aana,13.0,Pitched,NaN,Kathmandu,4500000.0,NaN,False,False,False,False,Budhanilkantha
1515,"Bhangal , Kathmandu",East,4300000.0,per_aana,16.0,Paved,NaN,Kathmandu,4300000.0,NaN,False,False,False,False,Bhangal
875,"Tinkune, Kathmandu",North,8500000.0,per_aana,25.0,Pitched,NaN,Kathmandu,8500000.0,NaN,False,False,False,False,Tinkune
1495,"Kirtipur, Kathmandu",East,2600000.0,per_aana,20.0,Paved,NaN,Kathmandu,2600000.0,NaN,False,False,False,False,Kirtipur
997,"Imadol, Lalitpur",East,5000000.0,per_aana,13.0,Gravel,4.1,Lalitpur,5000000.0,20500000.0,False,False,False,False,Imadol
698,"Lubhu, Lalitpur",North,3200000.0,per_aana,13.0,Pitched,3.2,Lalitpur,3200000.0,10240000.0,False,False,False,False,Lubhu
1207,"Balkot, Bhaktapur",East,3600000.0,per_aana,20.0,Gravel,NaN,Bhaktapur,3600000.0,NaN,False,False,False,False,Balkot
621,"Lubhu, Lalitpur",South-West,2800000.0,per_aana,20.0,Paved,NaN,Lalitpur,2800000.0,NaN,False,False,False,False,Lubhu


In [368]:
# Select all text columns
text_columns = ['location_raw', 'district', 'neighborhood', 'facing', 'road_type']

for col in text_columns:
    # Remove leading/trailing spaces and set to Title Case
    df2[col] = df2[col].astype(str).str.strip().str.title()
    
# Replace common 'null-like' strings with a proper NaN for pandas
df2[text_columns] = df2[text_columns].replace(['Nan', 'None', 'Unknown', 'Null', ''], np.nan)

In [371]:
# Ensure all flags are Boolean (True/False)
flag_cols = ['is_price_suspect', 'is_large_plot', 'is_rate_outlier', 'is_price_outlier', 'is_wide_road']

for col in flag_cols:
    if col in df2.columns:
        df2[col] = df2[col].astype(bool)

In [373]:
df2.columns

Index(['location_raw', 'facing', 'raw_price', 'price_unit', 'road_width_feet',
       'road_type', 'land_size_aana', 'district', 'price_per_aana',
       'calculated_total_price', 'is_price_suspect', 'is_large_plot',
       'is_rate_outlier', 'is_price_outlier', 'neighborhood'],
      dtype='object')

In [375]:
df2.sample(2)

,location_raw,facing,raw_price,price_unit,road_width_feet,road_type,land_size_aana,district,price_per_aana,calculated_total_price,is_price_suspect,is_large_plot,is_rate_outlier,is_price_outlier,neighborhood
1016,"Shantinagar, Kathmandu",West,7500000.0,per_aana,20.0,Paved,8.2,Kathmandu,7500000.0,61500000.0,False,False,False,False,Shantinagar
1693,"Lazimpat, Kathmandu",North,6500000.0,per_aana,13.0,Pitched,5.2,Kathmandu,6500000.0,33800000.0,False,False,False,False,Lazimpat


In [376]:
df2['is_wide_road'] = df2['road_width_feet'] > 40

In [377]:
export_columns = [
    'neighborhood', 'district', 'location_raw', 
    'price_per_aana', 'land_size_aana', 'calculated_total_price',
    'road_width_feet', 'road_type', 'facing',
    'is_wide_road', 'is_price_suspect', 'is_large_plot',
    'is_rate_outlier', 'is_price_outlier'
]

# Create the final dataframe version
df2_export = df2[export_columns].copy()

# Final export to CSV
df2_export.to_csv('cleaned_nepali_land_v2.csv', index=False)

print("Export Complete: 'cleaned_nepali_land_v2.csv' is ready.")

Export Complete: 'cleaned_nepali_land_v2.csv' is ready.


THIRD FILE CLEANING 

In [379]:
df3.sample(12)

,title,location_raw,floors,price_npr,land_size_aana,buildup_area_sqft,road_width_feet,facing,built_year_bs,built_year_ad,house_age,parking_cars,parking_bikes,bedrooms,bathrooms,amenities,district
479,"Residential house on sale in Bhangal, Budhanil...","Bhangal , Kathmandu",2.5,32000000.0,3.1,NaN,13.0,North,2078.0,2021.0,4.0,NaN,NaN,5.0,6.0,"Bathroom, Drinking Water, Parking, Drainage",Kathmandu
2639,Duplex house on urgent sale in Bhaisepati,"Bhaisepati, Lalitpur",2.5,25000000.0,4.0,NaN,13.0,East,2065.0,2008.0,17.0,NaN,NaN,6.0,4.0,"Power Backup, Drinking Water, Bathroom, Draina...",Lalitpur
406,Under construction House for Sale,"Thecho, Lalitpur",2.5,85000000.0,9.3,4706.0,12.0,South,2075.0,2018.0,7.0,1.0,2.0,6.0,5.0,"Earthquake Resistant, Marbel, Parquet, Drainag...",Lalitpur
1898,"Commercial house for rent at Jhamsikhel, Lalitpur","Jhamsikhel, Lalitpur",3.0,250000.0,15.0,NaN,17.0,NaN,2055.0,1998.0,27.0,NaN,NaN,NaN,3.0,"Bathroom, Drainage, Parking, Marbel",Lalitpur
1344,"House in Sale at Taukhel, Godawari","Taukhel, Lalitpur",2.5,30000000.0,4.2,NaN,8.0,North,2075.0,2018.0,7.0,NaN,NaN,4.0,3.0,"Bathroom, Drainage, Drinking Water, Marbel, Pa...",Lalitpur
2110,"House on sale at Gokul Awas, Bhaisepati","Bhaisepati, Lalitpur",2.5,45000000.0,4.2,NaN,20.0,South-East,2075.0,2018.0,7.0,NaN,NaN,4.0,5.0,"Drinking Water, Bathroom, Drainage, Power Back...",Lalitpur
418,Single Storey House for Sale,"Danchhi, Kathmandu",1.0,15000000.0,3.1,NaN,10.0,South,2060.0,2003.0,22.0,1.0,2.0,2.0,1.0,"Earthquake Resistant, Marbel, Drinking Water, ...",Kathmandu
3165,Syuchatar House,"Syuchatar, Kathmandu",2.5,22000000.0,4.0,NaN,14.0,North,2070.0,2013.0,12.0,1.0,3.0,5.0,2.0,"Lawn, Garage, Air Condition, Backyard, Balcony...",Kathmandu
582,House on sale in Mulpani,"Mulpani, Kathmandu",2.5,24500000.0,3.0,NaN,15.0,East,2074.0,2017.0,8.0,NaN,NaN,5.0,3.0,"Earthquake Resistant, Marbel, Bathroom, Drinki...",Kathmandu
2760,House on sale in Tokha,"Panchetar, Kathmandu",1.0,15500000.0,4.2,NaN,10.0,East,2079.0,2022.0,3.0,NaN,NaN,2.0,1.0,"Drinking Water, Bathroom, Drainage, Parking",Kathmandu


In [382]:
# 1. Standardize title for searching
df3['title_clean'] = df3['title'].astype(str).str.lower()

# 2. Define rental identifiers (keeping your list)
rent_keywords = ['rent', 'lease', 'भाडा', 'flat', 'room', 'office space', 'shutter']

# 3. Create the rental mask with your added improvements:
# - Catching regex errors (price < 10,000)
# - Catching rent keywords
# - Catching low prices (< 2,000,000)
is_rental = (
    (df3['title_clean'].apply(lambda x: any(k in x for k in rent_keywords))) | 
    (df3['price_npr'] < 2000000) | 
    (df3['price_npr'] < 10000)  # Handles those 0.03, 0.84 regex errors
)

# 4. Filter the dataframe to keep ONLY Sales
df3 = df3[~is_rental].copy()

# 5. Clean land_size_aana as planned (Sales shouldn't have 0.0 land)
df3['land_size_aana'] = df3['land_size_aana'].replace(0.0, np.nan)

# 6. Count and Check
print(f"Total Rental/Error Rows removed: {is_rental.sum()}")
print(f"Total Sale Rows remaining: {len(df3)}")

Total Rental/Error Rows removed: 599
Total Sale Rows remaining: 2629


In [383]:
df3.sample(3)

,title,location_raw,floors,price_npr,land_size_aana,buildup_area_sqft,road_width_feet,facing,built_year_bs,built_year_ad,house_age,parking_cars,parking_bikes,bedrooms,bathrooms,amenities,district,title_clean
1578,"House for sale in Jyotinagar, Budhanilkantha","Jyotinagar, Kathmandu",2.5,38800000.0,3.0,NaN,15.0,North,2068.0,2011.0,14.0,NaN,NaN,5.0,4.0,"Drinking Water, Bathroom, Drainage, Parking",Kathmandu,"house for sale in jyotinagar, budhanilkantha"
3074,SANOBHARYANG HOUSE FOR SALE,"SANOBHARYANG, KATHMANDU",2.5,49000000.0,4.2,NaN,20.0,South-East,2076.0,2019.0,6.0,1.0,3.0,5.0,2.0,NaN,Kathmandu,sanobharyang house for sale
1927,Beautiful house on sale in Dhapasi,"Dhapasi, Kathmandu",2.5,47000000.0,9.0,NaN,12.0,East,2075.0,2018.0,7.0,NaN,NaN,6.0,2.0,"Garden, Drainage, Power Backup, Bathroom, Drin...",Kathmandu,beautiful house on sale in dhapasi


In [384]:
# We split by comma and filter out empty strings to get an accurate count
df3['amenities'] = df3['amenities'].fillna('')
df3['amenity_count'] = df3['amenities'].apply(lambda x: len([i for i in x.split(',') if i.strip()]) if x != '' else 0)

In [386]:
from datetime import datetime

# 1. Get the current year dynamically
current_year = datetime.now().year

# 2. Normalize Year (BS to AD)
# AD = BS - 56.7 (approx 57 is standard for real estate data cleaning)
df3['built_year_ad'] = df3['built_year_ad'].fillna(df3['built_year_bs'] - 57)

# 3. Calculate House Age
df3['house_age'] = current_year - df3['built_year_ad']

# 4. Handle Future/Construction Dates
# Logic: If built year is in the future, it's under construction
df3['is_under_construction'] = df3['built_year_ad'] > current_year

# 5. Logical Clipping
# If age is negative (future build), set age to 0 for construction rows
# If age is extreme (e.g., > 80), it's likely a heritage site or data error
df3['house_age'] = df3['house_age'].apply(lambda x: 0 if x < 0 else (80 if x > 80 else x))

print(f"Calculated age using current year: {current_year}")

Calculated age using current year: 2026


In [387]:
df3.sample(2)

,title,location_raw,floors,price_npr,land_size_aana,buildup_area_sqft,road_width_feet,facing,built_year_bs,built_year_ad,house_age,parking_cars,parking_bikes,bedrooms,bathrooms,amenities,district,title_clean,amenity_count,is_under_construction
2363,"House on sale at Pasikot, Budhanilkantha","Pasikot, Kathmandu",2.5,38000000.0,5.00,NaN,15.0,East,2062.0,2005.0,21.0,NaN,NaN,4.0,4.0,"Drainage, Bathroom, Drinking Water, Parking, E...",Kathmandu,"house on sale at pasikot, budhanilkantha",5,False
1951,House on sale in Nakhipot Heights,"Nakhipot, Lalitpur",2.5,32500000.0,34.22,NaN,13.0,West,2055.0,1998.0,28.0,NaN,NaN,5.0,4.0,"Drinking Water, Bathroom, Drainage, Power Back...",Lalitpur,house on sale in nakhipot heights,7,False


In [390]:
# 1. Standardize numerical rooms
df3['bedrooms'] = pd.to_numeric(df3['bedrooms'], errors='coerce')
df3['bathrooms'] = pd.to_numeric(df3['bathrooms'], errors='coerce')

# 2. Commercial Suspect Flag
# If area is > 10,000 sqft or land is > 20 aana, it's likely commercial or a massive villa
df3['is_commercial_suspect'] = (df3['buildup_area_sqft'] > 10000) | (df3['land_size_aana'] > 20)

# 3. Neighborhood Extraction
# Splitting "Bhangal, Kathmandu" -> "Bhangal"
df3['neighborhood'] = df3['location_raw'].str.split(',').str[0].str.strip().str.title()

In [391]:
df3.sample(5)

,title,location_raw,floors,price_npr,land_size_aana,buildup_area_sqft,road_width_feet,facing,built_year_bs,built_year_ad,house_age,parking_cars,parking_bikes,bedrooms,bathrooms,amenities,district,title_clean,amenity_count,is_under_construction,is_commercial_suspect,neighborhood
1685,Beautiful house for sale at Imadol,"Imadol, Lalitpur",2.5,37500000.0,4.0,NaN,13.0,South,2079.0,2022.0,4.0,NaN,NaN,4.0,4.0,"Drainage, Marbel, Bathroom, Parking",Lalitpur,beautiful house for sale at imadol,4,False,False,Imadol
2878,House on sale at Nakhipot,"Nakhipot, Lalitpur",2.5,55000000.0,13.0,NaN,14.0,South,2079.0,2022.0,4.0,NaN,NaN,6.0,5.0,,Lalitpur,house on sale at nakhipot,0,False,False,Nakhipot
2933,"House on sale at Chandol, Baluwatar","Chandol, Kathmandu",NaN,34000000.0,6.2,NaN,NaN,NaN,2076.0,2019.0,7.0,NaN,NaN,NaN,NaN,"Parking, Lawn, Garage, Balcony, Backyard, Fron...",Kathmandu,"house on sale at chandol, baluwatar",16,False,False,Chandol
2755,Bungalow on sale in Budhanilkantha,"Budhanilkantha, Kathmandu",NaN,110000000.0,26.0,NaN,15.0,East,2078.0,2021.0,5.0,NaN,NaN,6.0,NaN,"Garden, Drainage, Bathroom, Drinking Water, Pa...",Kathmandu,bungalow on sale in budhanilkantha,5,False,True,Budhanilkantha
233,House for Sale,"Gongabu, Kathmandu",5.0,60000000.0,7.1,NaN,12.0,North,2076.0,2019.0,7.0,1.0,3.0,10.0,6.0,"Earthquake Resistant, Marbel, Drainage, Drinki...",Kathmandu,house for sale,11,False,False,Gongabu


In [392]:
df3['location_raw'] = df3['location_raw'].str.strip().str.title()

In [393]:
# 1. Wide Road Flag (Road > 60 feet)
df3['is_wide_road'] = df3['road_width_feet'] > 60

# 2. Commercial Suspect Flag
# Logic: If buildup area is massive OR land size is very large (> 20 aana)
df3['is_commercial_suspect'] = (df3['buildup_area_sqft'] > 10000) | (df3['land_size_aana'] > 20)

# 3. Clean up Land Size NaN
# Replacing 0.0 with NaN as planned, because a house sale must have land
df3['land_size_aana'] = df3['land_size_aana'].replace(0.0, np.nan)

In [394]:
df3.sample(5)

,title,location_raw,floors,price_npr,land_size_aana,buildup_area_sqft,road_width_feet,facing,built_year_bs,built_year_ad,house_age,parking_cars,parking_bikes,bedrooms,bathrooms,amenities,district,title_clean,amenity_count,is_under_construction,is_commercial_suspect,neighborhood,is_wide_road
165,Bungalow House for Sale,"Dhapasi, Kathmandu",3.0,47500000.0,7.0,NaN,20.0,East,2068.0,2011.0,15.0,2.0,5.0,4.0,5.0,"Earthquake Resistant, Parquet, Drainage, Power...",Kathmandu,bungalow house for sale,11,False,False,Dhapasi,False
1301,Brand new duplex house is on sale at Dhapakhel...,"Dhapakhel, Lalitpur",2.5,30000000.0,4.0,NaN,20.0,South-West,2079.0,2022.0,4.0,NaN,NaN,6.0,4.0,"Drinking Water, Bathroom, Drainage, Parking",Lalitpur,brand new duplex house is on sale at dhapakhel...,4,False,False,Dhapakhel,False
2102,House for sale at Manamaiju,"Manamaiju, Kathmandu",3.5,38000000.0,4.0,NaN,12.0,South,2078.0,2021.0,5.0,NaN,NaN,9.0,4.0,"Drainage, Bathroom, Marbel, Parking",Kathmandu,house for sale at manamaiju,4,False,False,Manamaiju,False
2076,"House for sale in Paknajol, Thamel","Paknajol, Kathmandu",3.0,72500000.0,9.0,NaN,10.0,North,2075.0,2018.0,8.0,NaN,NaN,6.0,4.0,"Parking, Drainage, Bathroom, Drinking Water, G...",Kathmandu,"house for sale in paknajol, thamel",5,False,False,Paknajol,False
2087,"House on sale in Dhapasi Height, Tokha","Dhapasi, Kathmandu",1.5,10000000.0,2.5,NaN,6.0,South,2078.0,2021.0,5.0,NaN,NaN,3.0,2.0,"Earthquake Resistant, Bathroom, Drinking Water...",Kathmandu,"house on sale in dhapasi height, tokha",5,False,False,Dhapasi,False


In [395]:
df3.columns

Index(['title', 'location_raw', 'floors', 'price_npr', 'land_size_aana',
       'buildup_area_sqft', 'road_width_feet', 'facing', 'built_year_bs',
       'built_year_ad', 'house_age', 'parking_cars', 'parking_bikes',
       'bedrooms', 'bathrooms', 'amenities', 'district', 'title_clean',
       'amenity_count', 'is_under_construction', 'is_commercial_suspect',
       'neighborhood', 'is_wide_road'],
      dtype='object')

In [396]:
df3['parking_cars'] = df3['parking_cars'].fillna(0).clip(upper=10)
df3['parking_bikes'] = df3['parking_bikes'].fillna(0).clip(upper=20)

In [397]:
df3['floors'] = pd.to_numeric(df3['floors'], errors='coerce').clip(lower=1, upper=10)

In [398]:
df3['facing'] = df3['facing'].str.strip().str.title().str.replace('-', ' ')

In [399]:
temp_df3=df3.copy()

In [404]:
import numpy as np
import pandas as pd

# 1. Define Constants
AANA_TO_SQFT = 342.25
COVERAGE_FACTOR = 0.7  # Assuming 70% of land is built upon per floor

# 2. Function to estimate Build-up Area
def estimate_area(row):
    # If the real value exists and isn't a weird outlier (e.g., < 100), keep it
    if pd.notna(row['buildup_area_sqft']) and row['buildup_area_sqft'] > 100:
        return row['buildup_area_sqft']
    
    # Otherwise, calculate: (Land in Sqft) * Floors * Coverage
    if pd.notna(row['land_size_aana']) and pd.notna(row['floors']):
        land_sqft = row['land_size_aana'] * AANA_TO_SQFT
        return round(land_sqft * row['floors'] * COVERAGE_FACTOR, 2)
    
    return np.nan

# 3. Apply the Estimation
temp_df3['calculated_area_sqft'] = temp_df3.apply(estimate_area, axis=1)

# 4. Create a flag to track where we 'guessed' the data
# This is useful for ML models to understand data quality
temp_df3['is_area_estimated'] = temp_df3['buildup_area_sqft'].isna()

# 5. Clean Parking (Assume NaN means 0 parking spaces)
# We clip at 10 and 20 to prevent extreme outliers from breaking the mean
temp_df3['parking_cars'] = temp_df3['parking_cars'].fillna(0).clip(upper=10)
temp_df3['parking_bikes'] = temp_df3['parking_bikes'].fillna(0).clip(upper=20)

# 6. Clean Floors
# Convert to numeric and cap at 10 (rare to find a 10+ story private house)
temp_df3['floors'] = pd.to_numeric(temp_df3['floors'], errors='coerce').clip(lower=1, upper=10)

# 7. Final Step: Drop the original messy column to keep the dataframe lean
temp_df3.drop(columns=['buildup_area_sqft'], inplace=True)

print(f"Build-up area successfully estimated for {temp_df3['is_area_estimated'].sum()} rows.")

Build-up area successfully estimated for 2081 rows.


In [405]:
temp_df3.sample(2)

,title,location_raw,floors,price_npr,land_size_aana,road_width_feet,facing,built_year_bs,built_year_ad,house_age,parking_cars,parking_bikes,bedrooms,bathrooms,amenities,district,title_clean,amenity_count,is_under_construction,is_commercial_suspect,neighborhood,is_wide_road,calculated_area_sqft,is_area_estimated
674,"Beautiful house on sale at Tyanglaphat, Kirtipur","Tyanglafat, Kathmandu",2.5,57000000.0,7.2,20.0,East,2079.0,2022.0,4.0,0.0,0.0,4.0,3.0,"Parking, Drainage, Bathroom, Garden",Kathmandu,"beautiful house on sale at tyanglaphat, kirtipur",4,False,False,Tyanglafat,False,4312.35,True
1315,House for sale in land price at Budhanilkantha,"Budhanilkantha, Kathmandu",2.5,46500000.0,7.0,15.0,West,2072.0,2015.0,11.0,0.0,0.0,9.0,6.0,"Bathroom, Drainage, Drinking Water, Parking",Kathmandu,house for sale in land price at budhanilkantha,4,False,False,Budhanilkantha,False,4192.56,True


In [407]:
df3=temp_df3.copy()

In [408]:
df3.sample(4)

,title,location_raw,floors,price_npr,land_size_aana,road_width_feet,facing,built_year_bs,built_year_ad,house_age,parking_cars,parking_bikes,bedrooms,bathrooms,amenities,district,title_clean,amenity_count,is_under_construction,is_commercial_suspect,neighborhood,is_wide_road,calculated_area_sqft,is_area_estimated
2008,"House for sale near Real Banquet, Bafal","Bafal, Kathmandu",2.5,36500000.0,4.0,12.0,South West,2063.0,2006.0,20.0,0.0,0.0,7.0,3.0,"Drainage, Bathroom, Drinking Water, Parking",Kathmandu,"house for sale near real banquet, bafal",4,False,False,Bafal,False,2400.00,False
1106,"Newly built house on sale at Bhaisepati, Lalitpur","Bhaisepati, Lalitpur",2.5,55000000.0,5.2,16.0,South East,2060.0,2003.0,23.0,0.0,0.0,5.0,4.0,"Earthquake Resistant, Marbel, Parquet, Fridge,...",Lalitpur,"newly built house on sale at bhaisepati, lalitpur",11,False,False,Bhaisepati,False,2850.00,False
1514,Booking open for luxurious house at Madhuban H...,"Deuwa Niwas, Kathmandu",2.5,41500000.0,4.0,17.0,North,2069.0,2012.0,14.0,0.0,0.0,4.0,4.0,"Modular Kitchen, Parking, Bathroom, Drinking W...",Kathmandu,booking open for luxurious house at madhuban h...,11,False,False,Deuwa Niwas,False,2161.00,False
556,"House on sale at Gairidhara, Lazimpat","Lazimpat, Kathmandu",2.5,40000000.0,4.1,12.0,North,2068.0,2011.0,15.0,0.0,0.0,NaN,NaN,"Drinking Water, Bathroom, Parking, Drainage",Kathmandu,"house on sale at gairidhara, lazimpat",4,False,False,Lazimpat,False,2455.64,True


In [416]:
# 1. Standardize Facing
# Cleaning up string artifacts and ensuring consistent casing
temp_df3['facing'] = temp_df3['facing'].str.strip().str.title().str.replace('-', ' ')

# 2. Boolean Encoding for Key Amenities
# We turn the messy 'amenities' text into binary columns (1 for Yes, 0 for No)
# This is a critical step for Machine Learning models to 'see' value
top_amenities = ['Modular Kitchen', 'Parquet', 'Drainage', 'Solar', 'Parking', 'Garden']

for item in top_amenities:
    col_name = f'has_{item.lower().replace(" ", "_")}'
    # We use .astype(int) so the model gets 1s and 0s instead of True/False
    temp_df3[col_name] = temp_df3['amenities'].str.contains(item, case=False, na=False).astype(int)

# 3. Calculate Normalized Rate (Price per Aana)
# Even for houses, this is the 'gold standard' metric for value in Nepal
temp_df3['price_per_aana'] = temp_df3['price_npr'] / temp_df3['land_size_aana']

# 4. Final Dataset Cleanup
# Dropping redundant or helper columns that we don't need anymore
cols_to_remove = ['title_clean', 'built_year_bs', 'amenities']
temp_df3 = temp_df3.drop(columns=[c for c in cols_to_remove if c in temp_df3.columns])

# 5. Handle any infinite rates from potential 0 land size errors
temp_df3['price_per_aana'] = temp_df3['price_per_aana'].replace([np.inf, -np.inf], np.nan)

print("temp_df3 is now fully polished and model-ready.")
print(f"Amenities extracted: {', '.join([f'has_{i.lower()}' for i in top_amenities])}")

KeyError: 'amenities'

In [417]:
temp_df3.columns

Index(['title', 'location_raw', 'floors', 'price_npr', 'land_size_aana',
       'road_width_feet', 'facing', 'built_year_ad', 'house_age',
       'parking_cars', 'parking_bikes', 'bedrooms', 'bathrooms', 'district',
       'amenity_count', 'is_under_construction', 'is_commercial_suspect',
       'neighborhood', 'is_wide_road', 'calculated_area_sqft',
       'is_area_estimated', 'has_modular_kitchen', 'has_parquet',
       'has_drainage', 'has_solar', 'has_parking', 'has_garden',
       'price_per_aana', 'ad_type', 'category'],
      dtype='object')

In [418]:
# 1. Add the 'type' column to distinguish from 'land' data
temp_df3['category'] = 'house'

# 2. Define the Final Column Order (Logical Grouping)
final_column_order = [
    # Identity & Category
    'category', 'title', 
    
    # Location
    'district', 'neighborhood', 'location_raw', 
    
    # Financials
    'price_npr', 'price_per_aana',
    
    # Land & Structure Basics
    'land_size_aana', 'calculated_area_sqft', 'floors', 'facing', 'road_width_feet',
    
    # Interior Details
    'bedrooms', 'bathrooms', 'parking_cars', 'parking_bikes', 
    
    # Age & Construction
    'built_year_ad', 'house_age', 'is_under_construction',
    
    # Amenities & Binary Features
    'amenity_count', 'has_modular_kitchen', 'has_parquet', 'has_drainage', 
    'has_solar', 'has_parking', 'has_garden',
    
    # Quality Flags
    'is_wide_road', 'is_commercial_suspect', 'is_area_estimated'
]

# 3. Rename columns for absolute clarity (Optional but recommended)
# I've kept most but made them very clean
rename_map = {
    'price_npr': 'total_price',
    'calculated_area_sqft': 'build_up_area',
    'land_size_aana': 'land_area_aana'
}

# Apply reordering and renaming
# We use [final_column_order] to rearrange the columns
temp_df3_final = temp_df3[final_column_order].rename(columns=rename_map)

print("Final House Dataset Ready.")
print(f"Columns are now ordered as: {temp_df3_final.columns.tolist()}")

Final House Dataset Ready.
Columns are now ordered as: ['category', 'title', 'district', 'neighborhood', 'location_raw', 'total_price', 'price_per_aana', 'land_area_aana', 'build_up_area', 'floors', 'facing', 'road_width_feet', 'bedrooms', 'bathrooms', 'parking_cars', 'parking_bikes', 'built_year_ad', 'house_age', 'is_under_construction', 'amenity_count', 'has_modular_kitchen', 'has_parquet', 'has_drainage', 'has_solar', 'has_parking', 'has_garden', 'is_wide_road', 'is_commercial_suspect', 'is_area_estimated']


In [420]:
temp_df3=temp_df3_final.copy()

In [421]:
temp_df3.sample(2)

,category,title,district,neighborhood,location_raw,total_price,price_per_aana,land_area_aana,build_up_area,floors,facing,road_width_feet,bedrooms,bathrooms,parking_cars,parking_bikes,built_year_ad,house_age,is_under_construction,amenity_count,has_modular_kitchen,has_parquet,has_drainage,has_solar,has_parking,has_garden,is_wide_road,is_commercial_suspect,is_area_estimated
1464,house,"Residential house for sale in Pepsicola, Kathm...",Kathmandu,Pepsicola,"Pepsicola, Kathmandu",26000000.0,8.666667e+06,3.0,2515.54,3.5,North,10.0,8.0,4.0,0.0,0.0,2012.0,14.0,False,4,0,0,1,0,1,0,False,False,True
10,house,House for Sale,Lalitpur,Imadol,"Imadol, Lalitpur",49900000.0,9.596154e+06,5.2,3737.37,3.0,North East,13.0,6.0,5.0,0.0,0.0,2022.0,4.0,False,11,0,1,0,0,1,0,False,False,True


In [422]:
temp_df3.to_csv('cleaned_nepali_houses_v2.csv', index=False)